In [1]:
# python 3.13.9

In [2]:
import numpy as np 
import pandas as pd
import os

In [3]:
data='/Users/Tysia/Desktop/Edukacja/Informatyka/Magisterka/ml-1m'
rating_data=os.path.join(data,'ratings.dat')
movie_data=os.path.join(data,'movies.dat')
user_data=os.path.join(data,'users.dat')

ratings=pd.read_csv(
  rating_data,
  sep='::',
  engine='python',
  names=['user','item','rating','timestamp'],
  encoding='latin-1'   
)

movies=pd.read_csv(
    movie_data, 
    sep='::',
    engine='python',
    names=['item', 'title', 'genres'],
    encoding='latin-1'
)

users = pd.read_csv(
    user_data, 
    sep='::',
    engine='python',
    names=['user', 'gender', 'age', 'occupation', 'zipcode'],
    encoding='latin-1'
)

In [3]:
print(ratings.head())
print(movies.head())
print(users.head())

   user  item  rating  timestamp
0     1  1193       5  978300760
1     1   661       3  978302109
2     1   914       3  978301968
3     1  3408       4  978300275
4     1  2355       5  978824291
   item                               title                        genres
0     1                    Toy Story (1995)   Animation|Children's|Comedy
1     2                      Jumanji (1995)  Adventure|Children's|Fantasy
2     3             Grumpier Old Men (1995)                Comedy|Romance
3     4            Waiting to Exhale (1995)                  Comedy|Drama
4     5  Father of the Bride Part II (1995)                        Comedy
   user gender  age  occupation zipcode
0     1      F    1          10   48067
1     2      M   56          16   70072
2     3      M   25          15   55117
3     4      M   45           7   02460
4     5      M   25          20   55455


In [ ]:
#  _ nie ma w NumPy 2.0. , więc przypisane są nowsze akcje
np.float_ = np.float64
np.int = int
np.bool = bool
np.complex_=np.complex128
np.unicode_=np.str_

In [5]:
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import ItemKNN, Pop, NeuMF, BPR
from recbole.trainer import Trainer
from recbole.utils import init_seed

In [ ]:
# przygotowanie danych
# wymaga specyficznych nazw kolumn (nazwa:typ)
# tworzenie kopii ratings, ze zmienionymi nazwami kolumn
recbole_df = ratings[['user', 'item', 'rating', 'timestamp']].copy()
recbole_df.columns = ['user_id:token', 'item_id:token', 'rating:float', 'timestamp:float']

In [ ]:
# def. ścieżki dla tymczasowych danych 
recbole_data_path = 'recbole_data'
dataset_name = 'ml-1m'
full_path = os.path.join(recbole_data_path, dataset_name)

# tworzymy katalog, jeśli nie istnieje
if not os.path.exists(full_path):
    os.makedirs(full_path)

# zapis pliku .inter, format wymagany przez bibliotekę
inter_file_path = os.path.join(full_path, f'{dataset_name}.inter')
recbole_df.to_csv(inter_file_path, index=False, sep='\t')

In [ ]:
# itemknn
config_dict = {
    'model': 'ItemKNN',
    'k': 20,                # liczba sąsiadów
    
    'dataset': dataset_name,
    'data_path': recbole_data_path,
    'field_separator': '\t', # separator użyty przy zapisie .inter
    'USER_ID_FIELD': 'user_id',
    'ITEM_ID_FIELD': 'item_id',
    'RATING_FIELD': 'rating',
    'LABEL_FIELD': 'rating',
    'load_col': {'inter': ['user_id', 'item_id', 'rating', 'timestamp']},
    
    # oodział danych: 80% trening, 0% walidacja, 20% testowanie 
    # rs - random split, losowy podział
    'eval_args': {
        'split': {'RS': [0.8, 0.0, 0.2]},
        'group_by': 'user',
        'order': 'RO',
        'mode': 'labeled'
    },
    
    'metrics': ['RMSE', 'MAE'],
  #  'topk': 10,              # top 10 rekomendacji
    'seed': 42,
    'show_progress': True  # pasek postępu 
}

In [ ]:
config = Config(model='ItemKNN', dataset=dataset_name, config_dict=config_dict)
# tworzenie datasetu z pliku
dataset = create_dataset(config)

c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the interm

In [10]:
# valid będzie pusty
train, valid, test = data_preparation(config, dataset)

In [12]:
iknn = ItemKNN(config, dataset).to(config['device'])
trainer = Trainer(config, iknn)
# trenowanie, obliczenie macierzy podobieństwa
trainer.fit(train)

(-inf, None)

In [13]:
result = trainer.evaluate(test, load_best_model=False)
print(result)

OrderedDict({'rmse': np.float64(3.2678), 'mae': 2.6159})


In [14]:
# mostpop
config_pop = {
    'model': 'Pop',
    
    'dataset': dataset_name,
    'data_path': recbole_data_path,
    'field_separator': '\t',
    'USER_ID_FIELD': 'user_id',
    'ITEM_ID_FIELD': 'item_id',
    'RATING_FIELD': 'rating',
    'load_col': {'inter': ['user_id', 'item_id', 'rating', 'timestamp']},
    
    'eval_args': {
        'split': {'RS': [0.8, 0.0, 0.2]},
        'group_by': 'user',
        'order': 'RO',
        'mode': 'full'
    },
    
    'metrics': ['Recall', 'NDCG', 'Precision'],
    'topk': 5,
    'seed': 2024,
    'show_progress': False
}

In [15]:
configPop = Config(model='Pop', dataset=dataset_name, config_dict=config_pop)
init_seed(configPop['seed'], configPop['reproducibility'])
datasetPop = create_dataset(configPop)

c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the interm

In [16]:
trainPop, validPop, testPop = data_preparation(configPop, datasetPop)

In [17]:
pop = Pop(configPop, datasetPop).to(configPop['device'])
trainer = Trainer(configPop, pop)
trainer.fit(trainPop)

c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\trainer\trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)


(-inf, None)

In [19]:
resultPop = trainer.evaluate(testPop,  load_best_model=False)
print(resultPop)

OrderedDict({'recall@5': np.float64(0.0406), 'ndcg@5': np.float64(0.2176), 'precision@5': np.float64(0.207)})


In [48]:
# NeuMF
config_neumf = {
    'model': 'NeuMF',
    
    # parametry  dla sieci neuronowej NeuMF
    'epochs': 20,                  # l. epok treningu
    'train_batch_size': 256,      # rozmiar partii danych
    'learner': 'adam',             # optymalizator
    'learning_rate': 0.001,        # szybkość uczenia
    'mf_embedding_size': 32,       
    'mlp_embedding_size': 32,      
    'mlp_hidden_size': [64, 32, 16, 8],  # warstwy ukryte sieci MLP
    'dropout_prob': 0.1,           # zapobiega przeuczeniu
    
    'dataset': dataset_name,
    'data_path': recbole_data_path,
    'field_separator': '\t',
    'USER_ID_FIELD': 'user_id',
    'ITEM_ID_FIELD': 'item_id',
    'RATING_FIELD': 'rating',
    'load_col': {'inter': ['user_id', 'item_id', 'rating', 'timestamp']},
    
    'eval_args': {
        'split': {'RS': [0.8, 0.0, 0.2]},
        'group_by': 'user',
        'order': 'RO',
        'mode': 'full'
    },
    
    'metrics': ['Recall', 'NDCG', 'Precision'],
    'topk': 5,
    'seed': 2024,
    'show_progress': True 
}

In [49]:
configNeumf = Config(model='NeuMF', dataset=dataset_name, config_dict=config_neumf)
init_seed(configNeumf['seed'], configNeumf['reproducibility'])
datasetNeumf = create_dataset(configNeumf)


c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the interm

In [50]:
trainNeumf, validNeumf, testNeumf = data_preparation(configNeumf, datasetNeumf)

In [51]:
neumf = NeuMF(configNeumf, datasetNeumf).to(configNeumf['device'])
trainer = Trainer(configNeumf, neumf)
trainer.fit(trainNeumf)

c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\trainer\trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)


(-inf, None)

In [52]:
resultNeumf = trainer.evaluate(testNeumf, load_best_model=False)
print(resultNeumf)

OrderedDict({'recall@5': np.float64(0.0631), 'ndcg@5': np.float64(0.2976), 'precision@5': np.float64(0.2885)})


In [ ]:
config_bpr = {
    'model': 'BPR',
    
    # parametry specyficzne dla BPR 
    'embedding_size': 10,    # wielkość wektorów/cech ukrytych użytkownika/przedmiotu
    'learning_rate': 0.05,  # szybkość uczenia
    'epochs': 30,            # l. epok
    'reg_weight': 0.0025,
    
    'dataset': dataset_name,
    'data_path': recbole_data_path,
    'field_separator': '\t',
    'USER_ID_FIELD': 'user_id',
    'ITEM_ID_FIELD': 'item_id',
    'RATING_FIELD': 'rating',
    'load_col': {'inter': ['user_id', 'item_id', 'rating', 'timestamp']},
    
    'eval_args': {
        'split': {'RS': [0.8, 0.0, 0.2]},
        'group_by': 'user',
        'order': 'RO',
        'mode': 'full'
    },
    
    'metrics': ['Recall', 'NDCG', 'Precision'],
    'topk': 5,
    'show_progress': True
}

In [43]:
configBpr = Config(model='BPR', dataset=dataset_name, config_dict=config_bpr)
init_seed(configBpr['seed'], configBpr['reproducibility'])

datasetBpr = create_dataset(configBpr)

c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the interm

In [44]:
trainBpr, validBpr, testBpr = data_preparation(configBpr, datasetBpr)

In [45]:
bprmf = BPR(configBpr, datasetBpr).to(configBpr['device'])
trainer = Trainer(configBpr, bprmf)
trainer.fit(trainBpr)

c:\Users\Tysia\AppData\Local\Programs\Python\Python313\Lib\site-packages\recbole\trainer\trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)


(-inf, None)

In [46]:
resultBpr = trainer.evaluate(testBpr, load_best_model=False)
print(resultBpr)

OrderedDict({'recall@5': np.float64(0.0403), 'ndcg@5': np.float64(0.2197), 'precision@5': np.float64(0.2138)})
